[![img/pythonista.png](img/pythonista.png)](https://www.pythonista.io)

# Combinación y remodelado de tablas con *Polars*.

Este *notebook* cubre dos temas que se apoyan en los *notebooks* 35 y 36:

1. **Combinación de tablas** (*joins*).
2. **Reestructuración** entre formato largo y ancho con `pivot()` y `unpivot()`.

Ambas operaciones ya se estudiaron con *pandas* (*notebooks* 14, 15 y 24); aquí se ve su equivalente en *Polars*. Al final, una sección de autoestudio presenta la combinación por la clave más cercana (*as-of join*).

El modo diferido, el motor *streaming*, los conjuntos de datos particionados, el *SQL* y la medición de rendimiento se estudian en el *notebook* 38.

* La siguiente celda importa *Polars* y configura la presentación de los números de punto flotante.

In [ ]:
import os

import polars as pl

pl.Config.set_fmt_float('full')
print(f"Polars {pl.__version__}")

* La siguiente celda lee los tres archivos de datos de ejemplo (descritos en el *notebook* 34) y calcula el importe neto de cada venta.

In [ ]:
ventas = (
    pl.read_csv(os.path.join('data', 'ventas.csv'), try_parse_dates=True)
    .with_columns(
        importe_neto=(
            pl.col('cantidad') * pl.col('precio_unitario') * (1 - pl.col('descuento').fill_null(0))
        ).round(2)
    )
)
tiendas = pl.read_csv(os.path.join('data', 'tiendas.csv'), try_parse_dates=True)
tipo_cambio = pl.read_csv(os.path.join('data', 'tipo_cambio.csv'), try_parse_dates=True)

print(f"ventas: {ventas.shape} · tiendas: {tiendas.shape} · tipo_cambio: {tipo_cambio.shape}")
tiendas

## Combinación de tablas.

[`df.join()`](https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.join.html) combina dos *dataframes* según los valores de una o más columnas clave.

```
df.join(<otro>, on=<claves>, how=<tipo>, validate=<relación>, coalesce=<booleano>)
```

Donde:

* `<otro>` es el *dataframe* de la derecha.
* `<claves>` son las columnas que deben coincidir. Si se llaman distinto en cada lado, se usan `left_on` y `right_on`.
* `<tipo>` es el tipo de combinación (ver la tabla siguiente). Por omisión es `'inner'`.
* `validate` es opcional y verifica la relación entre las claves: `'1:1'`, `'1:m'`, `'m:1'` o `'m:m'`. Si no se cumple, se desencadena una excepción, lo que evita duplicar renglones por accidente.
* `coalesce=True` combina las columnas clave de ambos lados en una sola. Es útil en `'full'`, donde por omisión se conservan las dos.

| `how=` | Renglones del resultado | Columnas |
| :--- | :--- | :--- |
| `'inner'` | Sólo los que tienen coincidencia en ambos lados. | De ambos. |
| `'left'` | Todos los de la izquierda; nulos donde no hay coincidencia. | De ambos. |
| `'right'` | Todos los de la derecha; nulos donde no hay coincidencia. | De ambos. |
| `'full'` | Todos los de ambos lados; nulos donde no hay coincidencia. | De ambos. |
| `'semi'` | Los de la izquierda que **sí** tienen coincidencia. | Sólo de la izquierda. |
| `'anti'` | Los de la izquierda que **no** tienen coincidencia. | Sólo de la izquierda. |
| `'cross'` | Todas las combinaciones (producto cartesiano). No usa claves. | De ambos. |

Los datos de ejemplo tienen dos inconsistencias intencionales que permiten ver la diferencia entre los tipos: algunas ventas tienen `tienda_id` 99, que no está en el catálogo, y las tiendas 13 y 14 no tienen ventas.

[Guía de *joins*](https://docs.pola.rs/user-guide/transformations/joins/)

**Ejemplos:**

* La siguiente celda combina las ventas con el catálogo de tiendas con `'inner'` y `'left'`. El *inner join* pierde las ventas de la tienda 99; el *left join* las conserva con nulos en las columnas del catálogo.

In [ ]:
tiendas_reducido = tiendas.select('tienda_id', 'nombre', 'region')

inner = ventas.join(tiendas_reducido, on='tienda_id', how='inner')
left = ventas.join(tiendas_reducido, on='tienda_id', how='left', validate='m:1')

print(f"Ventas: {ventas.height:,} · inner: {inner.height:,} · left: {left.height:,}")
left.filter(pl.col('region').is_null()).select('venta_id', 'tienda_id', 'nombre', 'region').head(3)

* La siguiente celda usa `'anti'` en ambos sentidos para encontrar las inconsistencias: ventas cuya tienda no existe en el catálogo y tiendas sin ventas.

In [ ]:
ventas_sin_tienda = ventas.join(tiendas, on='tienda_id', how='anti')
tiendas_sin_ventas = tiendas.join(ventas, on='tienda_id', how='anti')

print(f"Ventas con tienda inexistente: {ventas_sin_tienda.height} (tienda_id: {ventas_sin_tienda['tienda_id'].unique().to_list()})")
tiendas_sin_ventas.select('tienda_id', 'nombre', 'fecha_apertura')

* La siguiente celda usa `'semi'` para obtener las tiendas que vendieron artículos de electrónica el 31 de diciembre de 2025. A diferencia de un *inner join*, cada tienda aparece una sola vez, aunque tenga varias ventas, y sólo con sus propias columnas.

In [ ]:
electronica_31_dic = ventas.filter(
    pl.col('fecha_hora').dt.date() == pl.date(2025, 12, 31),
    pl.col('categoria') == 'Electrónica',
)
print(f"Ventas: {electronica_31_dic.height}")
tiendas.join(electronica_31_dic, on='tienda_id', how='semi').select('tienda_id', 'nombre', 'region')

* La siguiente celda combina el importe total por tienda con el catálogo usando `'full'` y `coalesce=True`: aparecen tanto la tienda 99 (sin datos de catálogo) como las tiendas 13 y 14 (sin importe).

In [ ]:
por_tienda = ventas.group_by('tienda_id').agg(pl.col('importe_neto').sum().round(2).alias('importe'))

por_tienda.join(tiendas_reducido, on='tienda_id', how='full', coalesce=True).sort('tienda_id').tail(4)

* La siguiente celda usa `'cross'` para generar todas las combinaciones de tienda y canal, y les une el importe de cada combinación. Así aparecen también las combinaciones sin ventas (las de las tiendas 13 y 14), que de otro modo faltarían en el resultado.

In [ ]:
canales = ventas.select('canal').unique()

importe_tienda_canal = ventas.group_by('tienda_id', 'canal').agg(
    pl.col('importe_neto').sum().round(2).alias('importe')
)

tiendas_reducido.join(canales, how='cross').join(
    importe_tienda_canal, on=['tienda_id', 'canal'], how='left'
).sort('tienda_id', 'canal').tail(6)

* La siguiente celda intenta combinar las ventas con un catálogo que tiene una tienda repetida, validando que la relación sea `'m:1'`. Se desencadenará una excepción de tipo `ComputeError` porque la clave no es única en el catálogo.

In [ ]:
catalogo_duplicado = pl.concat([tiendas_reducido, tiendas_reducido.head(1)])
ventas.join(catalogo_duplicado, on='tienda_id', how='left', validate='m:1')

## Formato largo y formato ancho.

* En **formato largo**, cada renglón es una observación: por ejemplo, una combinación de mes, categoría e importe. Es el formato más cómodo para filtrar, agrupar y graficar.
* En **formato ancho**, los valores de una columna se convierten en columnas: por ejemplo, un renglón por mes y una columna por categoría. Es el formato más cómodo para leer una tabla o compararla visualmente.

```
<df>.pivot(on=<columnas>, index=<índice>, values=<valores>, aggregate_function=<función>)
<df>.unpivot(on=<columnas>, index=<índice>, variable_name=<nombre_variable>, value_name=<nombre_valor>)
```

Donde, en [`pivot()`](https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.pivot.html) (de largo a ancho):

* `on` es la columna (o columnas) cuyos valores distintos se convierten en columnas nuevas.
* `index` es la columna (o columnas) que identifica cada renglón del resultado.
* `values` es la columna (o columnas) cuyos valores llenan las celdas.
* `aggregate_function` indica cómo se combinan los valores si hay varios para una misma celda: `'sum'`, `'mean'`, `'first'`, `'len'`, entre otras. Si se omite y hay varios valores, se desencadena una excepción.

Y en [`unpivot()`](https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.unpivot.html) (de ancho a largo):

* `on` son las columnas que se convierten en renglones. Por omisión, todas las que no están en `index`.
* `index` son las columnas que se conservan como identificadores.
* `variable_name` es el nombre de la columna que recibe los nombres de las columnas originales (por omisión, `'variable'`).
* `value_name` es el nombre de la columna que recibe los valores (por omisión, `'value'`).

Las columnas que produce `pivot()` dependen de los datos. Por eso, en el modo diferido (que se estudia en el *notebook* 38) `pivot()` exige indicar de antemano los valores que serán columnas con el argumento `on_columns`: el *schema* del resultado debe conocerse antes de ejecutar la consulta.

**Ejemplos:**

* La siguiente celda calcula el importe trimestral por categoría en formato largo.

In [ ]:
largo = (
    ventas
    .with_columns(trimestre=pl.col('fecha_hora').dt.year().cast(pl.String) + '-T' + pl.col('fecha_hora').dt.quarter().cast(pl.String))
    .group_by('trimestre', 'categoria')
    .agg(pl.col('importe_neto').sum().round(2).alias('importe'))
    .sort('trimestre', 'categoria')
)
largo.head(6)

* La siguiente celda convierte el resultado a formato ancho: un renglón por trimestre y una columna por categoría.

In [ ]:
ancho = largo.pivot(on='categoria', index='trimestre', values='importe', aggregate_function='sum')
ancho

* La siguiente celda regresa al formato largo con `unpivot()` y verifica que se obtienen los mismos datos.

In [ ]:
regreso = ancho.unpivot(index='trimestre', variable_name='categoria', value_name='importe')
regreso.sort('trimestre', 'categoria').equals(largo)

## 📘 Autoestudio: Combinación por la clave más cercana.

> **📘 Autoestudio.** Esta sección no se ve en clase y ningún otro *notebook* ni sección de clase depende de ella.
>
> * **Qué aprenderás:** combinar dos tablas cuando la clave no coincide exactamente, por ejemplo una venta con el último tipo de cambio publicado.
> * **Requisitos:** la sección «Combinación de tablas» de este *notebook* y las fechas del *notebook* 22.
> * **Tiempo estimado:** 10 minutos.
> * **Cómo saber si lo hiciste bien:** al final hay un ejercicio y una celda que comprueba tu resultado.

[`df.join_asof()`](https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.join_asof.html) combina cada renglón con el renglón de la otra tabla cuya clave es **la más cercana**, en lugar de exigir que sea igual. Es útil con datos de tiempo, por ejemplo para asignar a cada venta el último tipo de cambio publicado.

```
df.join_asof(<otro>, on=<clave>, strategy=<estrategia>, tolerance=<tolerancia>)
```

Donde:

* `<clave>` es una columna ordenable (generalmente una fecha). **Ambos *dataframes* deben estar ordenados por ella.**
* `<estrategia>` es `'backward'` (por omisión: el valor anterior o igual más cercano), `'forward'` (el posterior o igual) o `'nearest'`.
* `<tolerancia>` es opcional y limita la distancia máxima, por ejemplo `'3d'`.

**Ejemplo:**

El tipo de cambio sólo se publica en días hábiles. Con un *join* normal, las ventas de fines de semana quedarían sin tipo de cambio; con `join_asof()` y `strategy='backward'` toman el del viernes anterior.

* La siguiente celda asigna a cada venta el tipo de cambio vigente y muestra las ventas del viernes 5, el sábado 6 y el domingo 7 de enero de 2024 junto al tipo de cambio publicado ese día.

In [ ]:
ventas_usd = (
    ventas
    .with_columns(fecha=pl.col('fecha_hora').dt.date())
    .sort('fecha')
    .join_asof(tipo_cambio.sort('fecha'), on='fecha', strategy='backward')
    .with_columns(importe_usd=(pl.col('importe_neto') / pl.col('usd_mxn')).round(2))
)

# Ventas del sábado 6 y domingo 7 de enero de 2024: usan el tipo de cambio del viernes 5.
ventas_usd.filter(
    pl.col('fecha').is_between(pl.date(2024, 1, 5), pl.date(2024, 1, 7))
).group_by('fecha').agg(pl.col('usd_mxn').first(), pl.len()).sort('fecha').join(
    tipo_cambio, on='fecha', how='left', suffix='_publicado'
)

### Ejercicio.

Con `strategy='forward'`, cada venta recibe el **primer** tipo de cambio publicado en su fecha o después. Obtén `ventas_forward`: las ventas con las columnas `fecha` (la fecha de la venta, sin hora) y `usd_mxn` (el tipo de cambio asignado con `join_asof()` y `strategy='forward'`). Las ventas de un domingo deben recibir el tipo de cambio del lunes siguiente.

* La siguiente celda es para tu solución: sustituye `None` por el código que calcula `ventas_forward`.

In [ ]:
ventas_forward = None  # Sustituye None por tu solución.

**Comprueba tu resultado.**

* La siguiente celda comprueba tu resultado con `assert`. Si algo no coincide, muestra un mensaje que indica qué revisar; si todavía no resuelves el ejercicio, sólo lo indica.

In [ ]:
def comprobar_forward(resultado):
    assert isinstance(resultado, pl.DataFrame), 'El resultado debe ser un dataframe de Polars.'
    assert {'fecha', 'usd_mxn'} <= set(resultado.columns), 'Faltan las columnas fecha o usd_mxn.'
    assert resultado.height == ventas.height, 'Debe haber un renglón por venta.'
    lunes = tipo_cambio.filter(pl.col('fecha') == pl.date(2024, 1, 8))['usd_mxn'].item()
    domingo = resultado.filter(pl.col('fecha') == pl.date(2024, 1, 7))
    assert domingo.height > 0, 'No hay ventas del domingo 7 de enero de 2024; revisa la columna fecha.'
    assert (domingo['usd_mxn'] == lunes).all(), 'Las ventas del domingo 7 de enero de 2024 deben tener el tipo de cambio del lunes 8; revisa strategy.'
    print('Correcto.')


if ventas_forward is None:
    print('Ejercicio pendiente: completa la celda anterior y vuelve a ejecutar esta.')
else:
    comprobar_forward(ventas_forward)

**Solución.**

* La siguiente celda contiene una solución posible y la comprueba con la misma función. Su código está oculto: intenta resolver el ejercicio antes de mostrarlo (haz clic en la barra a la izquierda de la celda).

In [ ]:
ventas_forward = (
    ventas
    .with_columns(fecha=pl.col('fecha_hora').dt.date())
    .sort('fecha')
    .join_asof(tipo_cambio.sort('fecha'), on='fecha', strategy='forward')
)

comprobar_forward(ventas_forward)

## Resumen.

| Concepto | Descripción |
| :--- | :--- |
| ***Joins*** | `inner`, `left`, `right`, `full`, `semi`, `anti` y `cross`; `validate` detecta claves duplicadas. |
| **`pivot()` y `unpivot()`** | Conversión entre formato largo y ancho. |
| **📘 Autoestudio** | `join_asof()` combina con la clave más cercana; ambos lados deben estar ordenados por ella. |

El *notebook* 38 estudia el modo diferido y el trabajo con datos más grandes que la memoria.

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2026.</p>